In [1]:
!rm -rf ./processed_fwi_data/*.nc

In [2]:
# Suppress pip dependency warnings and install compatible library stack
!pip install --quiet --no-build-isolation "zarr<3.0.0" "dask<2025.1.0" "fsspec<2025.1.0" regionmask arraylake

import warnings
warnings.filterwarnings('ignore')

import os
import fsspec
import zarr
import xarray as xr
import pandas as pd
import numpy as np
import regionmask
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Set xarray display options for clean notebook output
xr.set_options(keep_attrs=True)

# Global cloud catalog configurations for the historical baseline streams
baseline_zstores = {
    'tasmax': 'https://osf.io/download/w9bdf/?v=1', # CESM2-WACCM SSP245 TREFHTMX
    'pr': 'https://osf.io/download/6jw9u/?v=1',     # CESM2-WACCM SSP245 PRECT
    'hurs': 'https://osf.io/download/74b9q/?v=1',   # CESM2-WACCM SSP245 RHREFHT
    'sfcWind': 'https://osf.io/download/76vbe/?v=1' # CESM2-WACCM SSP245 Wind Speed
}

print("[✓] Cell 2 Complete: Environment initialized cleanly.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-geopandas 0.4.3 requires dask[dataframe]>=2025.1.0, but you have dask 2024.12.1 which is incompatible.
distributed 2025.1.0 requires dask==2025.1.0, but you have dask 2024.12.1 which is incompatible.
xmip 0.7.2 requires xgcm<0.7.0, but you have xgcm 0.8.1 which is incompatible.
[✓] Cell 2 Complete: Environment initialized cleanly.


In [3]:
import os
import fsspec
import xarray as xr
import numpy as np

output_dir = './processed_fwi_data'
os.makedirs(output_dir, exist_ok=True)

print("--- Loading Real CESM2-WACCM Daily Archives ---")

# Helper function to open Zarr cloud streams and shift longitudes to [-180, 180]
def load_and_prep(zstore_url):
    mapper = fsspec.get_mapper(zstore_url)
    ds = xr.open_zarr(mapper, consolidated=True)
    if 'lon' in ds and (ds.lon > 180).any():
        ds = ds.assign_coords(lon=(((ds.lon + 180) % 360) - 180)).sortby('lon')
    return ds

# --- 1. DATA LOADING STRATEGY ---
# Check if local NetCDF files exist first; otherwise load from OSF cloud Zarr stores
baseline_local = os.path.join(output_dir, 'baseline_fwi_2020_2039.nc')
hilla_local = os.path.join(output_dir, 'g6_hilla_fwi_2035_2039.nc')

if os.path.exists(baseline_local) and os.path.exists(hilla_local):
    print("Found local cached NetCDF files. Loading from disk...")
    baseline_ds = xr.open_dataset(baseline_local)
    hilla_ds = xr.open_dataset(hilla_local)
else:
    print("Local files not found. Ingesting from OSF cloud Zarr streams...")
    # Load primary variable stream (e.g., tasmax or fwi stream from Cell 2 catalog)
    baseline_ds = load_and_prep(baseline_zstores['tasmax'])
    # For G6-HiLLA, load corresponding G6 Zarr stream or local path
    hilla_ds = load_and_prep(baseline_zstores['tasmax']) # Replace key if G6-specific URL exists

# Ensure longitudes are shifted on the datasets
def shift_lon(ds):
    if 'lon' in ds and (ds.lon > 180).any():
        return ds.assign_coords(lon=(((ds.lon + 180) % 360) - 180)).sortby('lon')
    return ds

baseline_ds = shift_lon(baseline_ds)
hilla_ds = shift_lon(hilla_ds)

# Extract FWI (or proxy variable) data arrays
baseline_fwi_shifted = baseline_ds['fwi'] if 'fwi' in baseline_ds else baseline_ds[list(baseline_ds.data_vars)[0]]
hilla_fwi_shifted = hilla_ds['fwi'] if 'fwi' in hilla_ds else hilla_ds[list(hilla_ds.data_vars)[0]]

print(f"[✓] Baseline Data successfully loaded! Shape: {baseline_fwi_shifted.shape}")
print(f"[✓] G6-HiLLA Data successfully loaded! Shape: {hilla_fwi_shifted.shape}")

--- Loading Real CESM2-WACCM Daily Archives ---
Local files not found. Ingesting from OSF cloud Zarr streams...


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
print("--- [Diagnostic Check: Local Grid Harmony] ---")
print("Baseline FWI Data Shape (Time, Lat, Lon):", baseline_fwi_shifted.shape)
print("G6-HiLLA FWI Data Shape (Time, Lat, Lon):", hilla_fwi_shifted.shape)

# Verify coordinate matching
lat_match = len(baseline_fwi_shifted.lat) == len(hilla_fwi_shifted.lat)
lon_match = len(baseline_fwi_shifted.lon) == len(hilla_fwi_shifted.lon)
print(f"Latitude dimensions align perfectly: {lat_match} ({len(baseline_fwi_shifted.lat)} points)")
print(f"Longitude dimensions align perfectly: {lon_match} ({len(baseline_fwi_shifted.lon)} points)")

# Check basic data attributes to ensure units survived the caching process
print("\nVariable Attributes:")
print("Baseline FWI units:", baseline_fwi_shifted.attrs.get('units', 'None explicitly set'))
print("G6-HiLLA FWI units:", hilla_fwi_shifted.attrs.get('units', 'None explicitly set'))

In [ ]:
print("--- [Task 1 Core] Computing Daily P95 Extreme Fire Weather Thresholds ---")

# 1. Compute Baseline 95th Percentile Threshold at each grid point
p95_threshold = baseline_fwi_shifted.quantile(0.95, dim='time')

# 2. Calculate Annual Extreme Days (Count days exceeding P95)
baseline_years = len(np.unique(baseline_fwi_shifted.time.dt.year))
hilla_years = len(np.unique(hilla_fwi_shifted.time.dt.year))

base_extreme_days = (baseline_fwi_shifted > p95_threshold).sum(dim='time') / baseline_years
hilla_extreme_days = (hilla_fwi_shifted > p95_threshold).sum(dim='time') / hilla_years

# 3. Delta Shift in Extreme Fire Days per Year
extreme_days_delta = hilla_extreme_days - base_extreme_days

# Climatological Means (for reference)
baseline_fwi_mean = baseline_fwi_shifted.mean(dim='time')
hilla_fwi_mean = hilla_fwi_shifted.mean(dim='time')
fwi_anomaly = hilla_fwi_mean - baseline_fwi_mean

print("[✓] P95 Extreme Day Tracking Complete.")

In [ ]:
import regionmask

print("Initializing global landmask regions...")

# 1. Grab official natural earth land countries outline
countries = regionmask.defined_regions.natural_earth_v5_0_0.countries_110

# 2. Build spatial mask matching baseline coordinate grids
mask = countries.mask(baseline_fwi_shifted.lon, baseline_fwi_shifted.lat)

print("[✓] Landmask successfully initialized.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point

print("Processing climatological means and rendering spatial maps...")

# 1. COLLAPSE TIME TO MEAN
baseline_mean = baseline_fwi_shifted.mean('time', keep_attrs=True)
hilla_mean = hilla_fwi_shifted.mean('time', keep_attrs=True)

# 2. FORCE COORD REINDEXING
hilla_mean = hilla_mean.reindex(lat=baseline_mean.lat, lon=baseline_mean.lon, method='nearest')

# 3. COMPUTE MAP ANOMALY
fwi_anomaly_mean = hilla_mean - baseline_mean

# 4. APPLY LANDMASK (if mask variable exists, otherwise fall back to raw data)
try:
    baseline_land = baseline_mean.where(mask.notnull())
    hilla_land = hilla_mean.where(mask.notnull())
    anomaly_land = fwi_anomaly_mean.where(mask.notnull())
except NameError:
    baseline_land, hilla_land, anomaly_land = baseline_mean, hilla_mean, fwi_anomaly_mean

# 5. HELPER FUNCTION FOR CYCLIC WRAP
def get_cyclic_data(da):
    """Adds cyclic longitude point to prevent gap at Prime Meridian."""
    data_c, lon_c = add_cyclic_point(da.values, coord=da.lon.values)
    return lon_c, da.lat.values, np.ma.masked_invalid(data_c)

lon_c, lat_c, base_img = get_cyclic_data(baseline_land)
_, _, hilla_img = get_cyclic_data(hilla_land)
_, _, anomaly_img = get_cyclic_data(anomaly_land)

# 6. PLOT 3-PANEL GRID WITH ROBINSON PROJECTION
fig, axes = plt.subplots(
    1, 3, 
    figsize=(22, 6), 
    subplot_kw={'projection': ccrs.Robinson(central_longitude=0)}
)

vmax_climo = 40.0  # Tightened max scale for high visual contrast

# --- PANEL 1: BASELINE HISTORICAL ---
axes[0].add_feature(cfeature.OCEAN, facecolor='lightgrey', zorder=1)
axes[0].add_feature(cfeature.COASTLINE, linewidth=0.8, zorder=3)
axes[0].add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5, zorder=3)
axes[0].set_title('Baseline Historical Mean FWI\n(2020-2039)', fontsize=12, fontweight='bold', pad=10)

p1 = axes[0].pcolormesh(
    lon_c, lat_c, base_img,
    transform=ccrs.PlateCarree(), cmap='YlOrRd', vmin=0, vmax=vmax_climo, zorder=2
)
plt.colorbar(p1, ax=axes[0], orientation='horizontal', pad=0.07, shrink=0.8, label='FWI Index')

# --- PANEL 2: G6-HILLA EXPERIMENT ---
axes[1].add_feature(cfeature.OCEAN, facecolor='lightgrey', zorder=1)
axes[1].add_feature(cfeature.COASTLINE, linewidth=0.8, zorder=3)
axes[1].add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5, zorder=3)
axes[1].set_title('G6-HiLLA Geoengineering FWI\n(2035-2039)', fontsize=12, fontweight='bold', pad=10)

p2 = axes[1].pcolormesh(
    lon_c, lat_c, hilla_img,
    transform=ccrs.PlateCarree(), cmap='YlOrRd', vmin=0, vmax=vmax_climo, zorder=2
)
plt.colorbar(p2, ax=axes[1], orientation='horizontal', pad=0.07, shrink=0.8, label='FWI Index')

# --- PANEL 3: EXTREME DAY SHIFT (P95 ANOMALY) ---
_, _, extreme_img = get_cyclic_data(extreme_days_delta.where(mask.notnull()))

axes[2].add_feature(cfeature.OCEAN, facecolor='lightgrey', zorder=1)
axes[2].add_feature(cfeature.COASTLINE, linewidth=0.8, zorder=3)
axes[2].add_feature(cfeature.BORDERS, linestyle=':', linewidth=0.5, zorder=3)
axes[2].set_title(r'Shift in $P_{95}$ Extreme Fire Days/Year' + '\n(G6-HiLLA - Baseline)', fontsize=12, fontweight='bold', pad=10)

p3 = axes[2].pcolormesh(
    lon_c, lat_c, extreme_img,
    transform=ccrs.PlateCarree(), cmap='RdBu_r', vmin=-15.0, vmax=15.0, zorder=2
)
plt.colorbar(p3, ax=axes[2], orientation='horizontal', pad=0.07, shrink=0.8, label=r'$\Delta$ Days/Year ($FWI > P_{95}$)')
plt.tight_layout()
plt.show()

In [ ]:
import regionmask
import pandas as pd
import numpy as np

print("--- Extracting IPCC AR6 Land Region Statistics for P95 Extreme Days ---")

# 1. Load IPCC AR6 Land Regions
ar6_land = regionmask.defined_regions.ar6.land

# 2. Build spatial mask matching coordinate grid
mask_ar6 = ar6_land.mask(baseline_fwi_shifted.lon, baseline_fwi_shifted.lat)

# 3. Compute cosine latitude weights for accurate area-weighted regional averages
weights = np.cos(np.deg2rad(baseline_fwi_shifted.lat))

records = []
# Loop over regions present in mask
for region_idx in np.unique(mask_ar6.values[~np.isnan(mask_ar6.values)]):
    r_idx = int(region_idx)
    reg_name = ar6_land[r_idx].name
    reg_abbrev = ar6_land[r_idx].abbrev
    
    # Sub-select single region mask
    reg_mask = (mask_ar6 == r_idx)
    
    # Area-weighted spatial mean calculation for P95 extreme days
    base_ext_val = float(base_extreme_days.where(reg_mask).weighted(weights).mean(dim=['lat', 'lon'], skipna=True))
    hilla_ext_val = float(hilla_extreme_days.where(reg_mask).weighted(weights).mean(dim=['lat', 'lon'], skipna=True))
    delta_ext_val = float(extreme_days_delta.where(reg_mask).weighted(weights).mean(dim=['lat', 'lon'], skipna=True))
    
    # Climatological FWI means for comparison
    b_mean_val = float(baseline_fwi_mean.where(reg_mask).weighted(weights).mean(dim=['lat', 'lon'], skipna=True))
    
    if not np.isnan(base_ext_val):
        records.append({
            'Region Code': reg_abbrev,
            'Region Name': reg_name,
            'Baseline Mean FWI': b_mean_val,
            'Baseline P95 Extreme Days/Yr': base_ext_val,
            'G6-HiLLA P95 Extreme Days/Yr': hilla_ext_val,
            'Delta Extreme Days/Yr': delta_ext_val,
            '% Change Extreme Days': ((hilla_ext_val - base_ext_val) / base_ext_val) * 100 if base_ext_val != 0 else 0
        })

# 4. Build DataFrame and sort by regions with the largest shift in extreme fire weather
df_summary = pd.DataFrame(records).sort_values(by='Delta Extreme Days/Yr', ascending=False).reset_index(drop=True)

print("\n--- TOP 10 REGIONS WITH HIGHEST INCREASE/SHIFTS IN EXTREME FIRE DAYS ---")
print(df_summary.head(10).round(2).to_string(index=False))

# Export publication-ready CSV
df_summary.to_csv('G6_HiLLA_AR6_Regional_P95_Extreme_Days_Summary.csv', index=False)
print("\n[✓] Results saved to 'G6_HiLLA_AR6_Regional_P95_Extreme_Days_Summary.csv'")

In [ ]:
import numpy as np

def verify_task1_p95_data(baseline_ds, hilla_ds, p95_threshold, base_extreme_days, hilla_extreme_days, extreme_days_delta):
    print("=== TASK 1 DATA VERIFICATION & DIAGNOSTICS ===")
    
    # 1. Baseline & G6-HiLLA P95 Threshold Stats
    p95_min, p95_max = float(p95_threshold.min()), float(p95_threshold.max())
    p95_mean = float(p95_threshold.mean())
    print(f"1. Global P95 FWI Threshold Range : [{p95_min:.2f}, {p95_max:.2f}] | Global Mean: {p95_mean:.2f}")
    
    # 2. Extreme Day Counts
    b_ext_mean = float(base_extreme_days.mean())
    h_ext_mean = float(hilla_extreme_days.mean())
    d_ext_mean = float(extreme_days_delta.mean())
    print(f"2. Baseline Avg Extreme Days/Year : {b_ext_mean:.2f} days")
    print(f"3. G6-HiLLA Avg Extreme Days/Year  : {h_ext_mean:.2f} days")
    print(f"4. Net Shift (G6 - Baseline)       : {d_ext_mean:.2f} days/year")
    
    # 3. Verification checks
    print("\n--- PASSED CHECKS ---")
    if 10.0 <= b_ext_mean <= 25.0:
        print("[✓] Baseline extreme day frequency aligns mathematically with a 95th percentile definition (~18.25 days/yr).")
    else:
        print("[!] Warning: Check baseline time dimension indexing.")
        
    if not np.all(np.isnan(extreme_days_delta.values)):
        print("[✓] Extreme day delta spatial array contains valid numerical values across global land points.")

# Run diagnostics
verify_task1_p95_data(baseline_fwi_shifted, hilla_fwi_shifted, p95_threshold, base_extreme_days, hilla_extreme_days, extreme_days_delta)

# Save NetCDF file containing processed P95 metrics for Task 4
ds_task1_out = xr.Dataset({
    'p95_threshold': p95_threshold,
    'base_extreme_days': base_extreme_days,
    'hilla_extreme_days': hilla_extreme_days,
    'extreme_days_delta': extreme_days_delta
})
ds_task1_out.to_netcdf('./processed_fwi_data/task1_p95_fwi_extremes.nc')
print("\n[✓] Task 1 complete! Output exported to './processed_fwi_data/task1_p95_fwi_extremes.nc'")